# Executing attack strategies

This file is responsible for executing each attack method against the classification models.
Make sure to first run the `1 - Training classification models.ipynb` file.

## Parameters

In [1]:
from utils.parameters import *

# Load datasets

In [ ]:
from utils.helpers.load_datasets import load_all_datasets

datasets = load_all_datasets(copies=n_copies, random_state=random_state)

# Creating the neural networks

In [3]:
from utils.helpers.load_networks import load_networks, network_types

for dataset_name, dataset in datasets.items():
    load_networks(
        dataset=dataset,
        dataset_name=dataset_name,
        batch_size=batch_size,
        n_hidden_layers=n_hidden_layers,
        n_neurons=n_neurons,
        dropout_prob_fnn=dropout_prob_fnn,
        dropout_prob_snn=dropout_prob_snn,
        learning_rate_fnn=learning_rate_fnn,
        learning_rate_snn=learning_rate_snn,
        random_state=random_state
    )

## Attacks
### Prepare the attack data

### Conduct attacks

In [6]:
from utils.helpers.conduct_attacks import get_attack_datasets, create_and_train_idsgan, evaluate_idsgan, create_and_train_genaal, evaluate_genaal, create_and_train_polytope, evaluate_polytope
from utils.helpers.load_networks import network_types

def conduct_attacks(dataset: dict, attack_trials: dict):
    n_trial = max(list(attack_trials.values())) if len(attack_trials) > 0 else 0
    
    results_partial = {}
    
    # Prepare attack datasets
    attack_train_df, attack_test_df = get_attack_datasets(dataset, frac=dataset["attack_frac"], train_size=attack_train_size, random_state=random_state + n_trial)

    for network_name in network_types:
        #
        # IDSGAN
        #
        
        attack_name = f"idsgan_{network_name}"        
        if attack_name not in attack_trials or attack_trials[attack_name] < n_trials['idsgan']:
            print(f"Conducting IDSGAN attack on {network_name.upper()} (Trial {attack_trials.get(attack_name, 0) + 1}/{n_trials['idsgan']})")
            
            idsgan = create_and_train_idsgan(
                train_df=attack_train_df,
                test_df=attack_test_df,
                dataset=dataset,
                dataset_name=dataset_name,
                network_name=network_name,
                functional_features=dataset["functional_features"],
                noise_dim = noise_dim,
                n_hidden_layers_generator = n_hidden_layers_generator,
                n_neurons_generator = n_neurons_generator,
                learning_rate_generator = learning_rate_generator,
                steps_generator = steps_generator,
                n_hidden_layers_discriminator = n_hidden_layers_discriminator,
                n_neurons_discriminator = n_neurons_discriminator,
                learning_rate_discriminator = learning_rate_discriminator,
                steps_discriminator = steps_discriminator,
                weight_clip = weight_clip,
                batch_size = batch_size,
                idsgan_epochs = idsgan_epochs
            )

            results_run = evaluate_idsgan(
                idsgan,
                attack_test_df,
                dataset,
                network_name,
                dataset["functional_features"]
            )

            print(f"IDSGAN {network_name.upper()} Trial Results: Accuracy: {results_run['accuracy']:.4f}, Average Distance: {results_run['average_distance']:.4f}")
            results_partial[attack_name] = results_run

        #
        # GenAAL
        #
         
        attack_name = f"genaal_{network_name}"
        if attack_name not in attack_trials or attack_trials[attack_name] < n_trials['gen_aal']:
            print(f"Conducting GenAAL attack on {network_name.upper()} (Trial {attack_trials.get(attack_name, 0) + 1}/{n_trials['gen_aal']})")   
            
            genaal = create_and_train_genaal(
                train_df=attack_train_df,
                test_df=attack_test_df,
                dataset=dataset,
                dataset_name=dataset_name,
                network_name=network_name,
                latent_dim = latent_dim,
                vae_hidden = vae_hidden,
                sids_hidden = sids_hidden,
                lambda_kl = lambda_kl,
                lambda_recon = lambda_recon,
                lambda_l2 = lambda_l2,
                lambda_label = lambda_label,
                gen_lr = gen_lr,
                sid_lr = sid_lr,
                batch_size = batch_size,
                max_iterations = max_iterations,
                candidate_pool_k = candidate_pool_k,
                nquery = nquery,
                pretrain_epochs = pretrain_epochs,
                sids_epochs = sids_epochs,
                gen_epochs = gen_epochs,
                label_query = label_query
            )

            results_run = evaluate_genaal(
                genaal,
                attack_test_df,
                dataset,
                network_name
            )

            print(f"GenAAL {network_name.upper()} Trial Results: Accuracy: {results_run['accuracy']:.4f}, Average Distance: {results_run['average_distance']:.4f}")
            results_partial[attack_name] = results_run

        #
        # Polytope Attack
        #
        
        attack_name = f"polytope_{network_name}"
        
        if attack_name not in attack_trials or attack_trials[attack_name] < n_trials['polytope']:
            print(f"Conducting Polytope attack on {network_name.upper()} (Trial {attack_trials.get(attack_name, 0) + 1}/{n_trials['polytope']})")
            
            polytope_attack = create_and_train_polytope(
                train_df=attack_train_df,
                test_df=attack_test_df,
                dataset=dataset,
                network_name=network_name,
                batch_size=batch_size,
                random_state=random_state + n_trial,
                n_rays=n_rays,
                step_size=step_size)

            results_run = evaluate_polytope(
                polytope_attack,
                attack_test_df,
                dataset,
                network_name,
                functional_features=dataset["functional_features"],
                move_inside=0
            )

            print(f"Polytope {network_name.upper()} Trial Results: Accuracy: {results_run['accuracy']:.4f}, Average Distance: {results_run['average_distance']:.4f}")
            results_partial[attack_name] = results_run
        
    return results_partial

In [7]:
import json
import numpy as np

def numpy_to_python(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, (np.integer, np.int64)):
        return int(obj)
    elif isinstance(obj, (np.floating, np.float64)):
        return float(obj)
    elif isinstance(obj, dict):
        return {k: numpy_to_python(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [numpy_to_python(v) for v in obj]
    else:
        return obj
    
def load_results_from_json(filename="attack_results.json"):
    try:
        with open(filename, "r") as f:
            results = json.load(f)
    except FileNotFoundError:
        results = {}
        
    return results
    
def save_results_to_json(results, filename="attack_results.json"):
    results_old = load_results_from_json(filename)
        
    # Merge old results with new results
    for dataset_name in results:
        if dataset_name not in results_old:
            results_old[dataset_name] = results[dataset_name]
        else:
            results_old[dataset_name].extend(results[dataset_name])
            # Remove duplicates
            runs_unique = []
            
            for run in results_old[dataset_name]:
                if run not in runs_unique:
                    runs_unique.append(run)
            results_old[dataset_name] = runs_unique

    results = results_old
    with open(filename, "w") as f:
        json.dump(numpy_to_python(results), f)

In [ ]:
results = load_results_from_json(filename=attack_results_filename)

for dataset_name in datasets:    
    if dataset_name not in results:
        results[dataset_name] = []
        
    while True:
        attack_trials = {}
        for trial in results[dataset_name]:
            for attack_name in trial.keys():
                if attack_name not in attack_trials:
                    attack_trials[attack_name] = 1
                attack_trials[attack_name] += 1
        
        results_trial = conduct_attacks(datasets[dataset_name], attack_trials)
        if len(results_trial) == 0:
            break
    
        results[dataset_name].append(results_trial)
        save_results_to_json(results, filename=attack_results_filename)

## Evaluate attack metrics

In [ ]:
import numpy as np

attack_metrics = {}

for dataset_name in results:
    dataset_basename = dataset_name.split('_copy')[0]
    
    if dataset_basename not in attack_metrics:
        attack_metrics[dataset_basename] = {}
        
    for execution in results[dataset_name]:
        for attack_name, metrics in execution.items():
            if attack_name not in attack_metrics[dataset_basename]:
                attack_metrics[dataset_basename][attack_name] = {
                    "accuracy": [],
                    "average_distance": []
                }
                
            attack_metrics[dataset_basename][attack_name]["accuracy"].append(metrics["accuracy"])
            attack_metrics[dataset_basename][attack_name]["average_distance"].append(metrics["average_distance"])
            
for dataset_name in attack_metrics:
    for attack_name in attack_metrics[dataset_name]:
        acc_values = attack_metrics[dataset_name][attack_name]["accuracy"]
        dist_values = attack_metrics[dataset_name][attack_name]["average_distance"]
        
        attack_metrics[dataset_name][attack_name]["accuracy_mean"] = np.mean(acc_values)
        attack_metrics[dataset_name][attack_name]["accuracy_std"] = np.std(acc_values)
        attack_metrics[dataset_name][attack_name]["average_distance_mean"] = np.mean(dist_values)
        attack_metrics[dataset_name][attack_name]["average_distance_std"] = np.std(dist_values)
        
        print(f"{dataset_name} - {attack_name}: Accuracy: {attack_metrics[dataset_name][attack_name]['accuracy_mean']:.4f} ± {attack_metrics[dataset_name][attack_name]['accuracy_std']:.4f}, Average Distance: {attack_metrics[dataset_name][attack_name]['average_distance_mean']:.4f} ± {attack_metrics[dataset_name][attack_name]['average_distance_std']:.4f}")

ton_iot - polytope_fnn: Accuracy: 0.9662 ± 0.0388, Average Distance: 1.4792 ± 0.9294
ton_iot - polytope_snn: Accuracy: 0.8918 ± 0.0572, Average Distance: 1.2358 ± 0.9934
ton_iot - idsgan_fnn: Accuracy: 0.5455 ± 0.4863, Average Distance: 67.2948 ± 86.1872
ton_iot - genaal_fnn: Accuracy: 0.1546 ± 0.2303, Average Distance: 19.9687 ± 31.8902
ton_iot - idsgan_snn: Accuracy: 0.7460 ± 0.4086, Average Distance: 107.6091 ± 137.9810
ton_iot - genaal_snn: Accuracy: 0.4200 ± 0.3845, Average Distance: 14.8661 ± 16.4430
bot_iot - polytope_fnn: Accuracy: 0.8883 ± 0.1190, Average Distance: 4.8901 ± 4.6114
bot_iot - polytope_snn: Accuracy: 0.8342 ± 0.2450, Average Distance: 3.4861 ± 0.3194
bot_iot - idsgan_fnn: Accuracy: 0.9062 ± 0.2915, Average Distance: 299.2496 ± 278.0529
bot_iot - genaal_fnn: Accuracy: 0.0337 ± 0.0662, Average Distance: 1.9627 ± 0.0700
bot_iot - idsgan_snn: Accuracy: 0.9629 ± 0.1760, Average Distance: 268.8366 ± 273.9020
bot_iot - genaal_snn: Accuracy: 0.1452 ± 0.3120, Average Dist